In [15]:
import torch, os, cv2, random, numpy as np
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader


In [16]:
def corrupt_image(img):
    def noise(i): return np.clip(i+np.random.normal(0,20,i.shape),0,255).astype(np.uint8)
    def blur(i): return cv2.GaussianBlur(i,(5,5),0)
    def lowres(i):
        h,w=i.shape[:2]
        return cv2.resize(cv2.resize(i,(w//3,h//3)),(w,h))
    return random.choice([noise,blur,lowres])(img)




In [17]:
class RAImageDataset(Dataset):
    def __init__(self, root_dir, img_size=224):
        self.paths=[os.path.join(root_dir,f) for f in os.listdir(root_dir)]

        self.transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size,img_size))
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self,idx):
        img=cv2.imread(self.paths[idx])
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

        corrupted=corrupt_image(img.copy())

        return self.transform(corrupted), self.transform(img)


In [18]:
DATA_PATH=r"D:\Image Recognition\data"
loader=DataLoader(RAImageDataset(DATA_PATH),batch_size=8,shuffle=True)
device="cuda" if torch.cuda.is_available() else "cpu"


In [19]:
class Block(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(a,b,3,1,1),nn.BatchNorm2d(b),nn.ReLU())
    def forward(self,x): return self.net(x)

class ProcessingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            Block(3,64),Block(64,64),
            nn.MaxPool2d(2),
            Block(64,128),Block(128,128),
            nn.Upsample(scale_factor=2),
            Block(128,64),
            nn.Conv2d(64,3,3,1,1),nn.Sigmoid())
    def forward(self,x): return self.net(x)


In [20]:
model=ProcessingNet().to(device)
model.load_state_dict(torch.load("baseline_model.pth",map_location=device))


<All keys matched successfully>

In [21]:
teacher=models.vgg16(weights=models.VGG16_Weights.DEFAULT).to(device)
teacher.eval()

normalize=transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\VICTUS/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100%|██████████| 528M/528M [11:58<00:00, 770kB/s]    


In [24]:
mse=nn.MSELoss()
ce=nn.CrossEntropyLoss()
opt=optim.Adam(model.parameters(),lr=1e-4)
lam=0.1


for epoch in range(10):

    epoch_loss = 0
    batches = 0

    pbar = tqdm(loader)

    for corrupted,clean in pbar:

        corrupted,clean = corrupted.to(device),clean.to(device)
        restored = model(corrupted)

        img_loss = mse(restored,clean)

        with torch.no_grad():
            label = teacher(normalize(clean)).argmax(1)

        pred = teacher(normalize(restored))
        recog_loss = ce(pred,label)

        loss = img_loss + lam*recog_loss

        opt.zero_grad()
        loss.backward()
        opt.step()

        # accumulate
        epoch_loss += loss.item()
        batches += 1

        # live display
        pbar.set_postfix(
            img=img_loss.item(),
            cls=recog_loss.item(),
            total=loss.item()
        )

    
    print(f"Epoch {epoch+1} Loss: {epoch_loss/batches:.6f}")


100%|██████████| 375/375 [05:33<00:00,  1.12it/s, cls=0.729, img=0.00158, total=0.0745] 


Epoch 1 Loss: 0.094535


100%|██████████| 375/375 [05:28<00:00,  1.14it/s, cls=0.527, img=0.00297, total=0.0556]


Epoch 2 Loss: 0.094037


100%|██████████| 375/375 [05:28<00:00,  1.14it/s, cls=0.786, img=0.00542, total=0.084]  


...

Epoch 8 Loss: 0.085708


100%|██████████| 375/375 [06:18<00:00,  1.01s/it, cls=1.94, img=0.00187, total=0.196]  


Epoch 9 Loss: 0.085159


100%|██████████| 375/375 [10:53<00:00,  1.74s/it, cls=0.955, img=0.00212, total=0.0976] 

Epoch 10 Loss: 0.085690


In [25]:
torch.save(model.state_dict(),"RA_model_vgg16.pth")
print("Saved ")


Saved 
